# Modern Computer Vision: Fundamentals & Transfer Learning

## Learning Objectives
* Understand image representations as Tensors, normalization, and data augmentation.
* Compare Convolutional Neural Networks (CNNs) vs. Vision Transformers (ViTs).
* Setup Keras 3 with JAX backend and implement a robust CPU fallback strategy.
* Fine-tune a pre-trained image classifier using **Keras 3 + JAX** on scientific/microscopy or social science data.

In [ ]:
# Install Keras 3 and Keras Hub
!pip install -q --upgrade keras-hub
!pip install -q --upgrade keras
!pip install -q tensorflow-datasets

In [ ]:
import os
# Force Keras to use JAX backend (Google's high-performance numerical computing library)
os.environ["KERAS_BACKEND"] = "jax"

import keras
import jax
import numpy as np
import matplotlib.pyplot as plt

# Check available devices and configure fallback settings
devices = jax.devices()
print("Available devices:", devices)

# Auto-detect CPU-only environments to run CPU Fallback mode
is_cpu_only = all(d.platform == "cpu" for d in devices)
if is_cpu_only:
    print("⚠️ WARNING: No accelerator detected. Running in CPU Fallback mode.")
    BATCH_SIZE = 8  # Keep batch size small to prevent CPU OOM / slow steps
    EPOCHS = 3      # Run fewer epochs for quick demonstration
    MODEL_PRESET = "efficientnet_b0_imagenet" # Lighter weight CNN model
else:
    print("⚡ Accelerator detected! Running on high-performance GPU/TPU backend.")
    BATCH_SIZE = 32
    EPOCHS = 5
    MODEL_PRESET = "vit_base_patch16_224" # Full-size Vision Transformer


## 1. Preparing the Dataset

For this demo, we will use the **Malaria cell image dataset** from TensorFlow Datasets (TFDS). It contains 27,558 microscopy images of cells that are either healthy (uninfected) or parasitized.

We will configure our data loader to load, resize, normalize, and augment the images.

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

print("Loading Malaria dataset split (80% train, 20% validation)...")
(train_ds, val_ds), ds_info = tfds.load(
    "malaria",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True
)

num_classes = ds_info.features["label"].num_classes
class_names = ds_info.features["label"].names
print(f"Classes: {class_names} (count: {num_classes})")


### Preprocessing & Data Augmentation

Images in raw datasets often have differing sizes. We resize them to a uniform shape (e.g., `(224, 224)`) and scale pixel values to `[0, 1]`.
We also use data augmentation (random flips and rotations) to make our classifier more robust.

In [ ]:
IMAGE_SIZE = (224, 224)

def preprocess_image(image, label):
    # Resize to model shape
    image = tf.image.resize(image, IMAGE_SIZE)
    # Rescale pixels to [0, 1]
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Construct a Keras Sequential layer for data augmentation
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.1)
])

# Build pipelines
train_pipeline = (
    train_ds
    .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_pipeline = (
    val_ds
    .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
# Display a few sample images
images, labels = next(iter(train_pipeline))

plt.figure(figsize=(10, 10))
for i in range(min(9, BATCH_SIZE)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy())
    plt.title(class_names[labels[i].numpy()])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 2. Model Architecture & Fine-tuning

Instead of training from scratch, we leverage a pre-trained model (Transfer Learning). Depending on our hardware, we loaded either a Convolutional Neural Network backbone (**EfficientNetV2**) or a Vision Transformer backbone (**ViT**).

JAX will compile our model with XLA to run extremely fast on GPU/TPUs.

In [ ]:
import keras_hub

print(f"Instantiating pre-trained backbone model: {MODEL_PRESET}")

if "vit" in MODEL_PRESET:
    model = keras_hub.models.ViTImageClassifier.from_preset(
        MODEL_PRESET,
        num_classes=num_classes,
        activation="softmax"
)
else:
    model = keras_hub.models.EfficientNetClassifier.from_preset(
        MODEL_PRESET,
        num_classes=num_classes,
        activation="softmax"
    )

model.summary()


In [ ]:
# Compile model
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(1e-4),
    metrics=["accuracy"]
)

# Run training loop
history = model.fit(
    train_pipeline,
    validation_data=val_pipeline,
    epochs=EPOCHS
)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss over Epochs")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy over Epochs")
plt.legend()

plt.tight_layout()
plt.show()


## 3. Generalizing to Other Domains (e.g. Social Science)

The exact same pipeline can classify any domain of images. For instance, in **Social Sciences**, researchers use this method to analyze public space dynamics (detecting crowds, identifying protest formats, classifying building architecture types, or grouping historical photos in archives).

To use your own custom dataset, structure your directories as follows:
```
data/
  protest/
    img_01.jpg
    ...
  march/
    img_02.jpg
    ...
```
Then, load it into a Keras Dataset with a single function:
```python
custom_train_ds = keras.utils.image_dataset_from_directory(
    "data/",
    image_size=(224, 224),
    batch_size=32
)
```